# 公开来源研究简报试点分析

**目的：** 描述试次是否通过、人工时间是否完整，以及哪些预登记配对可以比较。这里不做因果推断、显著性检验或自动发布效率结论。


## 设置与数据来源

真实记录路径为 `data/trials.csv`。若不存在，则读取空的 `trials.template.csv`。合成测试数据不在此笔记本中生成，也不会替代真实结果。可选环境变量 `PILOT_FONT_PATH` 指向本地字体；缺失时图表使用 Matplotlib 回退字体，轴标签保持英文。


In [1]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from analysis import analyze_trials, load_trials, make_figures

ROOT = Path.cwd()
REAL_DATA = ROOT / "data" / "trials.csv"
TEMPLATE = ROOT / "trials.template.csv"
DATA_PATH = REAL_DATA if REAL_DATA.exists() else TEMPLATE
FONT_PATH = os.environ.get("PILOT_FONT_PATH")
print(f"数据来源：{DATA_PATH}")


数据来源：/home/noldus/.worktrees/github-personal-brand/skills-evidence-pilot/analysis/research-brief/trials.template.csv


## 验证与描述性汇总

人工总时间仅在五个互斥阶段全部有记录时计算。失败和重试保留在逐次表；只有满足预登记规则的配对才计算差值。


In [2]:
trials = load_trials(DATA_PATH)
result = analyze_trials(trials)
print(f"状态：{result.status}")
print(f"总尝试：{result.counts['total']}；接受：{result.counts['accepted']}；失败：{result.counts['failed']}；未知：{result.counts['unknown']}")
complete = 0 if result.pairs.empty else int((result.pairs["comparison_status"] == "comparable").sum())
incomplete = len(result.pairs) - complete
print(f"完整配对：{complete}；不完整配对：{incomplete}")
print("缺失清单：")
for item in result.missing_items:
    print(f"- {item}")


状态：unmeasured
总尝试：0；接受：0；失败：0；未知：0
完整配对：0；不完整配对：0
缺失清单：
- 尚无试次记录


In [3]:
if result.attempts.empty:
    print("待采集：尚无真实试次，不生成数值图。")
else:
    display(result.attempts)
    display(result.pairs)
    figures = make_figures(result, font_path=FONT_PATH)
    for figure in figures.values():
        display(figure)
        plt.close(figure)


待采集：尚无真实试次，不生成数值图。


## 解读边界与下一步

先检查失败、未知和缺失项，再看完整配对的累计人工时间。少量个人数据只能支持本人的工作流决定。若要评估某一项 skill，下一轮应固定其他条件，只改变该 skill。
